[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [8]:
import torch
import torch.nn as nn
import math
from torch import Tensor

In [2]:
from torch_judge import hint
hint('cross_attention')


💡 Hint for Multi-Head Cross-Attention:
   Q from decoder (x_q), K/V from encoder (x_kv). Project, reshape to multi-head, compute scaled dot-product attention (no causal mask). Concat heads and project output.



In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.W_q,self.W_k,self.W_v,self.W_o = nn.Linear(d_model,d_model),nn.Linear(d_model,d_model),nn.Linear(d_model,d_model),nn.Linear(d_model,d_model)
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads
        

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # shape (batchsize,head_num,seq_length,d_head)
        batchsize,se_q,d_model = x_q.shape
        _,se_kv,_ = x_kv.shape
        q:torch.Tensor = self.W_q(x_q).view(batchsize,se_q,self.num_heads,self.d_head).transpose(1,2)
        k:torch.Tensor = self.W_k(x_kv).view(batchsize,se_kv,self.num_heads,self.d_head).transpose(1,2)
        v:torch.Tensor = self.W_v(x_kv).view(batchsize,se_kv,self.num_heads,self.d_head).transpose(1,2)
        
        scores = q @ k.transpose(-2,-1)
        scores = scores / math.sqrt(self.d_head)
        
        weight = torch.softmax(scores,dim=-1)
        # shape (batchsize,head_num,seq_length,d_head)
        context = weight @ v
        # contiguous 按照当前 shape 顺序重新搬动内存里的
        context = context.transpose(1,2).contiguous().view(batchsize,se_q,self.d_model)
        return self.W_o(context)
        
        
        

In [14]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [15]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.3ms)
  ✅ [2/4] Q and KV different lengths (1.8ms)
  ✅ [3/4] No causal mask — all KV affects all Q (25.6ms)
  ✅ [4/4] Gradient flow (34.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (62.9ms total)
  Progress saved. Run status() to see your dashboard.

